In [ ]:
import requests, unicodedata

def katharo(keimeno):
    # αφαιρεί τόνους και κάνει κεφαλαία, για να βρίσκουμε εύκολα λέξεις
    keimeno = unicodedata.normalize("NFD", keimeno)
    keimeno = "".join(c for c in keimeno if unicodedata.category(c) != "Mn")
    return keimeno.upper()

apantisi = requests.get("https://diavgeia.gov.gr/opendata/organizations.json")
foreis = apantisi.json()["organizations"]
print("Σύνολο φορέων στη Διαύγεια:", len(foreis))
print()

for f in foreis:
    onoma = katharo(f.get("label", ""))
    if "ΜΕΤΣΟΒΙΟ" in onoma or " ΕΜΠ" in onoma:
        print(f["uid"], "-", f["label"])

Σύνολο φορέων στη Διαύγεια: 5433

50323 - ΔΗΜΟΤΙΚΗ ΕΜΠΟΡΙΚΗ ΤΟΥΡΙΣΤΙΚΗ ΕΠΙΧΕΙΡΗΣΗ ΚΑΛΑΜΑΡΙΑΣ ΑΝΩΝΥΜΗ ΕΤΑΙΡΕΙΑ-Ο.Τ.Α. (Δ.Ε.Τ.Ε.Κ. Α.Ε.-Ο.Τ.Α.)
99200058 - ΕΛΛΗΝΙΚΗ ΕΤΑΙΡΕΙΑ ΕΠΕΝΔΥΣΕΩΝ ΚΑΙ ΕΞΩΤΕΡΙΚΟΥ ΕΜΠΟΡΙΟΥ Α.Ε.
50085 - ΤΑΜΕΙΟ ΠΡΟΝΟΙΑΣ ΑΞΙΩΜΑΤΙΚΩΝ ΕΜΠΟΡΙΚΟΥ ΝΑΥΤΙΚΟΥ
50086 - ΤΑΜΕΙΟ ΠΡΟΝΟΙΑΣ ΚΑΤΩΤΕΡΩΝ ΠΛΗΡΩΜΑΤΩΝ ΕΜΠΟΡΙΚΟΥ ΝΑΥΤΙΚΟΥ
52067 - ΤΑΜΕΙΟ ΕΠΙΚΟΥΡΙΚΗΣ ΑΣΦΑΛΙΣΗΣ ΠΡΟΣΩΠΙΚΟΥ ΙΔΡΥΜΑΤΩΝ ΕΜΠΟΡΙΚΟΥ ΝΑΥΤΙΚΟΥ (ΤΕΑΠΙΕΝ)
99203026 - ΕΘΝΙΚΟ ΜΕΤΣΟΒΙΟ ΠΟΛΥΤΕΧΝΕΙΟ
99221095 - ΟΡΓΑΝΙΣΜΟΣ ΕΞΩΤΕΡΙΚΟΥ ΕΜΠΟΡΙΟΥ
100018377 - ΕΛΛΗΝΙΚΗ ΣΥΝΟΜΟΣΠΟΝΔΙΑ ΕΜΠΟΡΙΟΥ ΚΑΙ ΕΠΙΧΕΙΡΗΜΑΤΙΚΟΤΗΤΑΣ  ΕΣΕΕ
100018380 - ΙΝΣΤΙΤΟΥΤΟ ΕΜΠΟΡΙΟΥ ΚΑΙ ΥΠΗΡΕΣΙΩΝ
100018385 - ΚΕΝΤΡΟ ΑΝΑΠΤΥΞΗΣ ΕΛΛΗΝΙΚΟΥ ΕΜΠΟΡΙΟΥ
100018471 - ΚΕΝΤΡΟ ΕΠΑΓΓΕΛΜΑΤΙΚΗΣ ΚΑΤΑΡΤΙΣΗΣ ΓΕΝΙΚΗΣ ΣΥΝΟΜΟΣΠΟΝΔΙΑΣ ΕΠΑΓΓΕΛΜΑΤΙΩΝ ΒΙΟΤΕΧΝΩΝ ΕΜΠΟΡΩΝ ΕΛΛΑΔΑΣ
100018486 - ΙΝΣΤΙΤΟΥΤΟ ΜΙΚΡΩΝ ΕΠΙΧΕΙΡΗΣΕΩΝ ΓΕΝΙΚΗΣ ΣΥΝΟΜΟΣΠΟΝΔΙΑΣ ΕΠΑΓΓΕΛΜΑΤΙΩΝ ΒΙΟΤΕΧΝΩΝ ΕΜΠΟΡΩΝ ΕΛΛΑΔΟΣ ΙΜΕΓΣΕΒΕΕ
100018648 - ΟΜΟΣΠΟΝΔΙΑ ΕΜΠΟΡΙΚΩΝ ΣΥΛΛΟΓΩΝ ΘΕΣΣΑΛΙΑΣ
100019207 - ΜΕΝΤΙΓΟΥΕΙ ΕΙΣΑΓΩΓΗ ΚΑΙ ΕΜΠΟΡΙΑ ΙΑΤΡΙΚΩΝ ΝΟΣΟΚ

In [ ]:
from huggingface_hub import list_repo_files
from google.colab import userdata

token = userdata.get("HF_TOKEN")
arxeia = list_repo_files("glossAPI/diavgeia", repo_type="dataset", token=token)

print("Πλήθος αρχείων:", len(arxeia))
print()
for a in arxeia[:40]:
    print(a)
print("...")
for a in arxeia[-15:]:
    print(a)

Πλήθος αρχείων: 19

.gitattributes
README.md
data/2010.parquet
data/2011.parquet
data/2012.parquet
data/2013.parquet
data/2014.parquet
data/2015.parquet
data/2016.parquet
data/2017.parquet
data/2018.parquet
data/2019.parquet
data/2020.parquet
data/2021.parquet
data/2022.parquet
data/2023.parquet
data/2024.parquet
data/2025.parquet
data/2026.parquet
...
data/2012.parquet
data/2013.parquet
data/2014.parquet
data/2015.parquet
data/2016.parquet
data/2017.parquet
data/2018.parquet
data/2019.parquet
data/2020.parquet
data/2021.parquet
data/2022.parquet
data/2023.parquet
data/2024.parquet
data/2025.parquet
data/2026.parquet


In [ ]:
import json, time, numpy as np, pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc
from datetime import datetime, timezone
from huggingface_hub import HfFileSystem
from google.colab import userdata, drive

# 1. Σύνδεση με το Google Drive (για να σωθεί το αρχείο)
FAKELOS = "/content/diavgeia_emp"
import os; os.makedirs(FAKELOS, exist_ok=True)

# 2. Σύνδεση με το Hugging Face
fs = HfFileSystem(token=userdata.get("HF_TOKEN"))
EMP = "99203026"   # ο κωδικός του ΕΜΠ στη Διαύγεια

def apofaseis_emp(xronia):
    arxeio = f"datasets/glossAPI/diavgeia/data/{xronia}.parquet"
    pf = pq.ParquetFile(fs.open(arxeio, "rb", block_size=16 * 1024 * 1024))
    plithos_omadon = pf.num_row_groups
    print(f"Έτος {xronia}: {pf.metadata.num_rows:,} αποφάσεις σε {plithos_omadon} κομμάτια")
    grammes, arxi = [], time.time()

    for i in range(plithos_omadon):
        # διαβάζουμε πρώτα μόνο τα στοιχεία (όχι το μεγάλο κείμενο)
        t = pf.read_row_group(i, columns=["id", "metadata_json"])
        maska = pc.match_substring(t["metadata_json"], EMP).to_numpy(zero_copy_only=False)
        thesis = np.nonzero(maska)[0]
        if len(thesis) > 0:
            keimena = pf.read_row_group(i, columns=["markdown_text"]).take(thesis)
            for k, j in enumerate(thesis):
                meta = json.loads(t["metadata_json"][int(j)].as_py())
                if str(meta.get("organizationId")) != EMP:
                    continue
                hm = meta.get("issueDate")
                grammes.append({
                    "ada": t["id"][int(j)].as_py(),
                    "imerominia": datetime.fromtimestamp(hm / 1000, timezone.utc).date() if hm else None,
                    "thema": meta.get("subject"),
                    "typos": meta.get("decisionTypeId"),
                    "extra": json.dumps(meta.get("extraFieldValues"), ensure_ascii=False),
                    "keimeno": (keimena["markdown_text"][k].as_py() or "")[:6000],
                })
        if (i + 1) % 10 == 0 or i + 1 == plithos_omadon:
            lepta = (time.time() - arxi) / 60
            print(f"  κομμάτι {i+1}/{plithos_omadon} — βρέθηκαν {len(grammes)} αποφάσεις ΕΜΠ — {lepta:.1f} λεπτά")

    df = pd.DataFrame(grammes)
    df.to_csv(f"{FAKELOS}/emp_{xronia}.csv", index=False, encoding="utf-8-sig")
    print(f"✅ Αποθηκεύτηκε: emp_{xronia}.csv ({len(df)} αποφάσεις)")
    return df

df = apofaseis_emp(2025)
df.head()

Έτος 2025: 2,973,302 αποφάσεις σε 260 κομμάτια
  κομμάτι 10/260 — βρέθηκαν 5 αποφάσεις ΕΜΠ — 0.0 λεπτά
  κομμάτι 20/260 — βρέθηκαν 25 αποφάσεις ΕΜΠ — 0.0 λεπτά
  κομμάτι 30/260 — βρέθηκαν 1175 αποφάσεις ΕΜΠ — 0.2 λεπτά
  κομμάτι 40/260 — βρέθηκαν 3033 αποφάσεις ΕΜΠ — 0.6 λεπτά
  κομμάτι 50/260 — βρέθηκαν 4638 αποφάσεις ΕΜΠ — 0.9 λεπτά
  κομμάτι 60/260 — βρέθηκαν 6503 αποφάσεις ΕΜΠ — 1.2 λεπτά
  κομμάτι 70/260 — βρέθηκαν 8004 αποφάσεις ΕΜΠ — 1.4 λεπτά
  κομμάτι 80/260 — βρέθηκαν 11979 αποφάσεις ΕΜΠ — 1.8 λεπτά
  κομμάτι 90/260 — βρέθηκαν 13536 αποφάσεις ΕΜΠ — 2.1 λεπτά
  κομμάτι 100/260 — βρέθηκαν 15513 αποφάσεις ΕΜΠ — 2.5 λεπτά
  κομμάτι 110/260 — βρέθηκαν 17265 αποφάσεις ΕΜΠ — 2.8 λεπτά
  κομμάτι 120/260 — βρέθηκαν 17406 αποφάσεις ΕΜΠ — 3.1 λεπτά
  κομμάτι 130/260 — βρέθηκαν 17409 αποφάσεις ΕΜΠ — 3.2 λεπτά
  κομμάτι 140/260 — βρέθηκαν 18257 αποφάσεις ΕΜΠ — 3.5 λεπτά
  κομμάτι 150/260 — βρέθηκαν 20543 αποφάσεις ΕΜΠ — 3.9 λεπτά
  κομμάτι 160/260 — βρέθηκαν 22128 αποφάσεις ΕΜΠ — 4.1 λεπτ

,ada,imerominia,thema,typos,extra,keimeno
0,ΡΞ5Ε46ΨΖΣ4-ΕΦΘ,2025-04-11,Αποφαση Προέδρου 3047 για δαπάνη στο έργο 6243...,Β.2.1,"{""org"": {""afm"": ""099793475"", ""afmType"": ""EL"", ...",ΕΘΝΙΚΟ ΜΕΤΣΟΒΙΟ ΠΟΛΥΤΕΧΝΕΙΟ\nΕΙΔΙΚΟΣ ΛΟΓΑΡΙΑΣΜ...
1,Ρ1ΘΓ46ΨΖΣ4-ΧΜ8,2025-04-14,Έγκριση αιτήματος μετακίνησης για το έργο 6140...,2.4.7.1,"{""kanonistikipraxitype"": null, ""kanonistikiPra...",ΕΘΝΙΚΟ ΜΕΤΣΟΒΙΟ ΠΟΛΥΤΕΧΝΕΙΟ\nΕΙΔΙΚΟΣ ΛΟΓΑΡΙΑΣΜ...
2,9ΦΑΠ46ΨΖΣ4-ΣΙΡ,2025-04-14,Έγκριση αιτήματος μετακίνησης για το έργο 6140...,2.4.7.1,"{""kanonistikipraxitype"": null, ""kanonistikiPra...",ΕΘΝΙΚΟ ΜΕΤΣΟΒΙΟ ΠΟΛΥΤΕΧΝΕΙΟ\nΕΙΔΙΚΟΣ ΛΟΓΑΡΙΑΣΜ...
3,996Τ46ΨΖΣ4-ΗΤΕ,2025-04-15,Έγκριση αιτήματος μετακίνησης για το έργο 6140...,2.4.7.1,"{""kanonistikipraxitype"": null, ""kanonistikiPra...",ΕΘΝΙΚΟ ΜΕΤΣΟΒΙΟ ΠΟΛΥΤΕΧΝΕΙΟ\nΕΙΔΙΚΟΣ ΛΟΓΑΡΙΑΣΜ...
4,ΡΤΦ546ΨΖΣ4-ΤΩ1,2025-04-15,Έγκριση αιτήματος μετακίνησης για το έργο 6326...,2.4.7.1,"{""kanonistikipraxitype"": null, ""kanonistikiPra...",ΕΘΝΙΚΟ ΜΕΤΣΟΒΙΟ ΠΟΛΥΤΕΧΝΕΙΟ\nΕΙΔΙΚΟΣ ΛΟΓΑΡΙΑΣΜ...


In [ ]:
typos_kathara = df["typos"].astype(str).str.replace("B", "Β")

print("ΠΟΣΕΣ ΑΠΟΦΑΣΕΙΣ ΑΝΑ ΤΥΠΟ:")
print(typos_kathara.value_counts().head(12))
print()

print("ΠΟΣΕΣ ΑΠΟΦΑΣΕΙΣ ΑΝΑ ΜΗΝΑ:")
print(df["imerominia"].astype(str).str[:7].value_counts().sort_index())
print()

anatheseis = df[typos_kathara == "Β.2.1"]
print("ΠΑΡΑΔΕΙΓΜΑ ΑΝΑΘΕΣΗΣ (Β.2.1):")
print("Θέμα:", anatheseis.iloc[0]["thema"])
print(json.dumps(json.loads(anatheseis.iloc[0]["extra"]), ensure_ascii=False, indent=2))

ΠΟΣΕΣ ΑΠΟΦΑΣΕΙΣ ΑΝΑ ΤΥΠΟ:
typos
Β.2.2      13952
Β.2.1       4201
Β.1.1       4179
Γ.3.4       4076
Β.1.3       3434
2.4.7.1     2432
Α.2          207
Δ.1          129
Γ.3.1         99
Γ.2           67
100           53
Γ.3.3         46
Name: count, dtype: int64

ΠΟΣΕΣ ΑΠΟΦΑΣΕΙΣ ΑΝΑ ΜΗΝΑ:
imerominia
2025-04      25
2025-05    3750
2025-06    6192
2025-07    6780
2025-08     890
2025-09    6914
2025-10      99
2025-11     540
2025-12    7798
Name: count, dtype: int64

ΠΑΡΑΔΕΙΓΜΑ ΑΝΑΘΕΣΗΣ (Β.2.1):
Θέμα: Αποφαση Προέδρου 3047 για δαπάνη στο έργο 62433400    #36492
{
  "org": {
    "afm": "099793475",
    "afmType": "EL",
    "name": "ΕΙΔΙΚΟΣ ΛΟΓΑΡΙΑΣΜΟΣ  ΚΟΝΔΥΛΙΩΝ ΕΡΕΥΝΑΣ ΕΘΝΙΚΟΥ  ΜΕΤΣΟΒΙΟΥ ΠΟΛΥΤΕΧΝΕΙΟΥ",
    "enterName": false
  },
  "sponsor": [
    {
      "sponsorAFMName": {
        "afm": "801122617",
        "name": "PIXTRICKS ΟΕ||PIXTRICKS ΟΕ",
        "enterName": false
      },
      "expenseAmount": {
        "amount": 744.0,
        "currency": "EUR"
      }
    }
  ],
  "relate

In [ ]:
from google.colab import files

def epexergasia(xronia):
    arxeio = f"datasets/glossAPI/diavgeia/data/{xronia}.parquet"
    pf = pq.ParquetFile(fs.open(arxeio, "rb", block_size=16 * 1024 * 1024))
    n = pf.num_row_groups
    print(f"Έτος {xronia}: {n} κομμάτια")
    apofaseis, keimena, arxi = [], [], time.time()

    for i in range(n):
        t = pf.read_row_group(i, columns=["id", "metadata_json"])
        maska = pc.match_substring(t["metadata_json"], EMP).to_numpy(zero_copy_only=False)
        thesis = np.nonzero(maska)[0]
        if len(thesis) == 0:
            continue
        kt = pf.read_row_group(i, columns=["markdown_text"]).take(thesis)
        for k, j in enumerate(thesis):
            meta = json.loads(t["metadata_json"][int(j)].as_py())
            if str(meta.get("organizationId")) != EMP:
                continue
            extra = meta.get("extraFieldValues") or {}
            if not isinstance(extra, dict):
                extra = {}
            typos = str(meta.get("decisionTypeId")).replace("B", "Β")

            # ποσό και κύριος ανάδοχος
            sponsors = extra.get("sponsor") or []
            poso, kyrios, megisto = 0.0, {}, -1
            for s in sponsors:
                try:
                    a = float((s.get("expenseAmount") or {}).get("amount") or 0)
                except (TypeError, ValueError):
                    a = 0.0
                poso += a
                if a > megisto:
                    megisto, kyrios = a, (s.get("sponsorAFMName") or {})
            anadoxos = (kyrios.get("name") or "").split("||")[0].strip()

            hm = meta.get("issueDate")
            ada = t["id"][int(j)].as_py()
            apofaseis.append({
                "ada": ada,
                "imerominia": datetime.fromtimestamp(hm / 1000, timezone.utc).date() if hm else None,
                "thema": meta.get("subject"),
                "typos": typos,
                "monada": (extra.get("org") or {}).get("name"),
                "anadoxos": anadoxos or None,
                "afm_anadoxou": kyrios.get("afm"),
                "poso": poso if sponsors else None,
            })
            if typos in ("Β.2.1", "Δ.1"):
                keimena.append({"ada": ada, "keimeno": (kt["markdown_text"][k].as_py() or "")[:3000]})

    print(f"  ✔ {len(apofaseis)} αποφάσεις σε {(time.time() - arxi) / 60:.1f} λεπτά")
    return apofaseis, keimena

oles_apofaseis, ola_keimena = [], []
for xronia in [2022, 2023, 2024, 2025, 2026]:
    a, k = epexergasia(xronia)
    oles_apofaseis += a
    ola_keimena += k

df_apof = pd.DataFrame(oles_apofaseis).drop_duplicates("ada")
df_keim = pd.DataFrame(ola_keimena).drop_duplicates("ada")
df_apof.to_parquet(f"{FAKELOS}/emp_apofaseis.parquet", index=False)
df_keim.to_parquet(f"{FAKELOS}/emp_keimena.parquet", index=False)

print()
print(f"✅ ΤΕΛΟΣ! {len(df_apof):,} αποφάσεις, {len(df_keim):,} κείμενα αναθέσεων/συμβάσεων")
print("Αναθέσεις ανά έτος και συνολικό ποσό (€):")
anath = df_apof[df_apof["typos"] == "Β.2.1"].copy()
anath["etos"] = pd.to_datetime(anath["imerominia"]).dt.year
print(anath.groupby("etos")["poso"].agg(["count", "sum"]).round(0))

files.download(f"{FAKELOS}/emp_apofaseis.parquet")
files.download(f"{FAKELOS}/emp_keimena.parquet")

Έτος 2022: 366 κομμάτια
  ✔ 56456 αποφάσεις σε 8.7 λεπτά
Έτος 2023: 353 κομμάτια
  ✔ 50177 αποφάσεις σε 7.5 λεπτά
Έτος 2024: 371 κομμάτια
  ✔ 64881 αποφάσεις σε 9.7 λεπτά
Έτος 2025: 260 κομμάτια
  ✔ 32988 αποφάσεις σε 4.7 λεπτά
Έτος 2026: 24 κομμάτια
  ✔ 9036 αποφάσεις σε 1.0 λεπτά

✅ ΤΕΛΟΣ! 213,538 αποφάσεις, 29,234 κείμενα αναθέσεων/συμβάσεων
Αναθέσεις ανά έτος και συνολικό ποσό (€):
      count         sum
etos                   
2022   8336  18298009.0
2023   7688  16913605.0
2024   7728  18458660.0
2025   4201  10502128.0
2026    455    840314.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>